# Store Sales Time Series Forecasting
## An End-to-End Forecasting Project

### Business Context
Corporación Favorita is a large Ecuadorian grocery retailer operating 54 stores 
across multiple cities. Accurate sales forecasting helps the business make better 
inventory decisions, reduce waste, and optimize staffing. This project builds a 
time series forecasting model to predict future sales for a specific store and 
product family.

## Project Overview

This project follows a complete time series forecasting workflow:

1. Data Collection and Exploration
2. Data Cleaning and Transformation
3. Exploratory Data Analysis
4. Model Development
5. Model Optimization
6. Streamlit Web Application Deployment
7. Documentation and Version Control

## 1. Data Collection and Exploration
### 1.1 Importing Libraries

In [1]:
# Import core libraries for data manipulation
import pandas as pd
import numpy as np

print("Libraries imported successfully")

Libraries imported successfully


### 1.2 Loading the Datasets

In [2]:
# Load all datasets
train = pd.read_csv('train.csv', parse_dates=['date'])
stores = pd.read_csv('stores.csv')
oil = pd.read_csv('oil.csv', parse_dates=['date'])
holidays = pd.read_csv('holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('transactions.csv', parse_dates=['date'])

print("All datasets loaded successfully")
print("Train shape:", train.shape)
print("Stores shape:", stores.shape)
print("Oil shape:", oil.shape)
print("Holidays shape:", holidays.shape)
print("Transactions shape:", transactions.shape)

All datasets loaded successfully
Train shape: (3000888, 6)
Stores shape: (54, 5)
Oil shape: (1218, 2)
Holidays shape: (350, 6)
Transactions shape: (83488, 3)


In [3]:
# Display the first few rows of each dataset
print("=== TRAIN ===")
print(train.head())
print("\n=== STORES ===")
print(stores.head())
print("\n=== OIL ===")
print(oil.head())
print("\n=== HOLIDAYS ===")
print(holidays.head())
print("\n=== TRANSACTIONS ===")
print(transactions.head())

=== TRAIN ===
   id       date  store_nbr      family  sales  onpromotion
0   0 2013-01-01          1  AUTOMOTIVE    0.0            0
1   1 2013-01-01          1   BABY CARE    0.0            0
2   2 2013-01-01          1      BEAUTY    0.0            0
3   3 2013-01-01          1   BEVERAGES    0.0            0
4   4 2013-01-01          1       BOOKS    0.0            0

=== STORES ===
   store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4

=== OIL ===
        date  dcoilwtico
0 2013-01-01         NaN
1 2013-01-02       93.14
2 2013-01-03       92.97
3 2013-01-04       93.12
4 2013-01-07       93.20

==

In [4]:
# Check the date range of our training data
print("Train date range:")
print("Start:", train['date'].min())
print("End:", train['date'].max())
print("Total days:", (train['date'].max() - train['date'].min()).days)

print("\nUnique stores:", train['store_nbr'].nunique())
print("Unique product families:", train['family'].nunique())
print("\nProduct families:")
print(train['family'].unique())

Train date range:
Start: 2013-01-01 00:00:00
End: 2017-08-15 00:00:00
Total days: 1687

Unique stores: 54
Unique product families: 33

Product families:
['AUTOMOTIVE' 'BABY CARE' 'BEAUTY' 'BEVERAGES' 'BOOKS' 'BREAD/BAKERY'
 'CELEBRATION' 'CLEANING' 'DAIRY' 'DELI' 'EGGS' 'FROZEN FOODS' 'GROCERY I'
 'GROCERY II' 'HARDWARE' 'HOME AND KITCHEN I' 'HOME AND KITCHEN II'
 'HOME APPLIANCES' 'HOME CARE' 'LADIESWEAR' 'LAWN AND GARDEN' 'LINGERIE'
 'LIQUOR,WINE,BEER' 'MAGAZINES' 'MEATS' 'PERSONAL CARE' 'PET SUPPLIES'
 'PLAYERS AND ELECTRONICS' 'POULTRY' 'PREPARED FOODS' 'PRODUCE'
 'SCHOOL AND OFFICE SUPPLIES' 'SEAFOOD']


In [5]:
# Check total sales by family to confirm GROCERY I is the highest
family_sales = train.groupby('family')['sales'].sum().sort_values(ascending=False)
print("Top 10 product families by total sales:")
print(family_sales.head(10))

Top 10 product families by total sales:
family
GROCERY I        3.434627e+08
BEVERAGES        2.169545e+08
PRODUCE          1.227047e+08
CLEANING         9.752129e+07
DAIRY            6.448771e+07
BREAD/BAKERY     4.213395e+07
POULTRY          3.187600e+07
MEATS            3.108647e+07
PERSONAL CARE    2.459205e+07
DELI             2.411032e+07
Name: sales, dtype: float64


In [6]:
# Filter data for Store 1 and GROCERY I
df = train[(train['store_nbr'] == 1) & (train['family'] == 'GROCERY I')].copy()

# Set date as index
df = df.set_index('date')

# Keep only relevant columns
df = df[['sales', 'onpromotion']]

print("Filtered dataset shape:", df.shape)
print("\nDate range:")
print("Start:", df.index.min())
print("End:", df.index.max())
print("\nFirst few rows:")
print(df.head())

Filtered dataset shape: (1684, 2)

Date range:
Start: 2013-01-01 00:00:00
End: 2017-08-15 00:00:00

First few rows:
             sales  onpromotion
date                           
2013-01-01     0.0            0
2013-01-02  2652.0            0
2013-01-03  2121.0            0
2013-01-04  2056.0            0
2013-01-05  2216.0            0


### 1.3 Initial Data Exploration

In [7]:
# Basic information about our filtered dataset
print("=== DATASET INFO ===")
print(df.info())

print("\n=== STATISTICAL SUMMARY ===")
print(df.describe())

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== ZERO SALES DAYS ===")
print("Number of days with zero sales:", (df['sales'] == 0).sum())
print("Percentage of zero sales days:", round((df['sales'] == 0).sum() / len(df) * 100, 2), "%")

=== DATASET INFO ===
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1684 entries, 2013-01-01 to 2017-08-15
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sales        1684 non-null   float64
 1   onpromotion  1684 non-null   int64  
dtypes: float64(1), int64(1)
memory usage: 39.5 KB
None

=== STATISTICAL SUMMARY ===
             sales  onpromotion
count  1684.000000  1684.000000
mean   2223.172803    17.560570
std     779.285352    24.741069
min       0.000000     0.000000
25%    1873.500000     0.000000
50%    2283.500000     6.000000
75%    2649.000000    28.000000
max    9065.000000   167.000000

=== MISSING VALUES ===
sales          0
onpromotion    0
dtype: int64

=== ZERO SALES DAYS ===
Number of days with zero sales: 6
Percentage of zero sales days: 0.36 %


In [8]:
# Identify zero sales days
zero_sales_days = df[df['sales'] == 0]
print("Days with zero sales:")
print(zero_sales_days)

Days with zero sales:
            sales  onpromotion
date                          
2013-01-01    0.0            0
2014-01-01    0.0            0
2015-01-01    0.0            0
2015-07-07    0.0            0
2016-01-01    0.0            0
2017-01-01    0.0            0


In [9]:
# Check if 2015-07-07 is in our holidays file
print(holidays[holidays['date'] == '2015-07-07'])

Empty DataFrame
Columns: [date, type, locale, locale_name, description, transferred]
Index: []


In [10]:
# Check for missing dates in our time series
full_date_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
missing_dates = full_date_range.difference(df.index)
print("Missing dates in time series:")
print(missing_dates)
print("Total missing dates:", len(missing_dates))

Missing dates in time series:
DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[ns]', freq=None)
Total missing dates: 4


### 1.4 Checking Data Types

In [12]:
# Confirm date index is proper datetime type
print("Index type:", type(df.index))
print("Index dtype:", df.index.dtype)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nDate range confirmation:")
print("Start:", df.index.min())
print("End:", df.index.max())
print("Frequency:", pd.infer_freq(df.index))

Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Index dtype: datetime64[ns]

Column dtypes:
sales          float64
onpromotion      int64
dtype: object

Date range confirmation:
Start: 2013-01-01 00:00:00
End: 2017-08-15 00:00:00
Frequency: None


### 1.5 Exploration Summary

Key findings from initial exploration:

- Dataset covers 1,684 days from January 2013 to August 2017 for Store 1, GROCERY I
- Average daily sales are $2,223 with a maximum of $9,065
- 6 days have zero sales, all corresponding to store closure days
- 4 dates are completely missing from the dataset, all Christmas Days
- Time series frequency is irregular due to missing dates
- No missing values in sales or onpromotion columns
- Oil prices, holidays, and transactions data will be merged as additional features

## 2. Data Cleaning and Transformation
### 2.1 Handling Missing Dates

In [13]:
# Reindex to fill missing dates with NaN
df = df.reindex(full_date_range)

print("Shape after reindexing:", df.shape)
print("\nMissing values after reindexing:")
print(df.isnull().sum())
print("\nNew missing dates filled with NaN:")
print(df[df['sales'].isnull()])


Shape after reindexing: (1688, 2)

Missing values after reindexing:
sales          4
onpromotion    4
dtype: int64

New missing dates filled with NaN:
            sales  onpromotion
2013-12-25    NaN          NaN
2014-12-25    NaN          NaN
2015-12-25    NaN          NaN
2016-12-25    NaN          NaN


In [14]:
# Fill missing Christmas dates with 0 as stores were closed
df['sales'] = df['sales'].fillna(0)
df['onpromotion'] = df['onpromotion'].fillna(0)

print("Missing values after filling:")
print(df.isnull().sum())
print("\nShape:", df.shape)
print("\nChristmas dates after filling:")
print(df[df.index.month == 12][df[df.index.month == 12].index.day == 25])

Missing values after filling:
sales          0
onpromotion    0
dtype: int64

Shape: (1688, 2)

Christmas dates after filling:
            sales  onpromotion
2013-12-25    0.0          0.0
2014-12-25    0.0          0.0
2015-12-25    0.0          0.0
2016-12-25    0.0          0.0


### 2.2 Handling Zero Sales Days

In [15]:
# Instead of removing zero sales days, we will interpolate them
# Zero sales on holidays should be replaced with interpolated values
# so the model does not learn that sales drop to zero on these days

# Identify zero sales days
zero_days = df[df['sales'] == 0].index
print("Zero sales days:")
print(zero_days)

# Replace zeros with NaN temporarily for interpolation
df['sales'] = df['sales'].replace(0, np.nan)

# Interpolate using linear method
df['sales'] = df['sales'].interpolate(method='linear')

print("\nSales after interpolation on previously zero days:")
print(df.loc[zero_days, 'sales'])

Zero sales days:
DatetimeIndex(['2013-01-01', '2013-12-25', '2014-01-01', '2014-12-25',
               '2015-01-01', '2015-07-07', '2015-12-25', '2016-01-01',
               '2016-12-25', '2017-01-01'],
              dtype='datetime64[ns]', freq=None)

Sales after interpolation on previously zero days:
2013-01-01       NaN
2013-12-25    2544.0
2014-01-01    1965.5
2014-12-25    2704.0
2015-01-01    1743.0
2015-07-07    2262.5
2015-12-25    2373.0
2016-01-01    2044.5
2016-12-25    2256.5
2017-01-01    1993.0
Name: sales, dtype: float64


In [17]:
# 2013-01-01 is the first row so interpolation could not fill it
# We fill it with the next available value using bfill
df['sales'] = df['sales'].bfill()

print("Missing values remaining:")
print(df.isnull().sum())
print("\n2013-01-01 sales after backfill:")
print(df.loc['2013-01-01', 'sales'])

Missing values remaining:
sales          0
onpromotion    0
dtype: int64

2013-01-01 sales after backfill:
2652.0


### 2.3 Outlier Detection and Treatment

In [18]:
from scipy import stats

# Calculate Z-scores for sales
z_scores = np.abs(stats.zscore(df['sales']))

# Identify outliers beyond 3 standard deviations
outliers = df[z_scores > 3]
print("Number of outliers detected:", len(outliers))
print("\nOutlier dates and sales values:")
print(outliers['sales'].sort_values(ascending=False))

Number of outliers detected: 12

Outlier dates and sales values:
2016-04-18    9065.0
2016-04-19    8221.0
2016-04-20    5438.0
2016-12-23    5386.0
2016-12-21    5162.0
2016-12-22    5055.0
2017-03-07    4917.0
2016-04-22    4871.0
2016-04-21    4768.0
2016-11-09    4669.0
2016-12-13    4624.0
2017-06-20    4575.0
Name: sales, dtype: float64


In [19]:
# Cross reference outliers with holidays
print("Checking April 2016 in holidays file:")
print(holidays[(holidays['date'] >= '2016-04-15') & 
               (holidays['date'] <= '2016-04-25')])

print("\nChecking December 2016 in holidays file:")
print(holidays[(holidays['date'] >= '2016-12-20') & 
               (holidays['date'] <= '2016-12-25')])

Checking April 2016 in holidays file:
          date     type    locale locale_name                description  \
219 2016-04-16    Event  National     Ecuador           Terremoto Manabi   
220 2016-04-17    Event  National     Ecuador         Terremoto Manabi+1   
221 2016-04-18    Event  National     Ecuador         Terremoto Manabi+2   
222 2016-04-19    Event  National     Ecuador         Terremoto Manabi+3   
223 2016-04-20    Event  National     Ecuador         Terremoto Manabi+4   
224 2016-04-21  Holiday     Local    Riobamba  Cantonizacion de Riobamba   
225 2016-04-21    Event  National     Ecuador         Terremoto Manabi+5   
226 2016-04-22    Event  National     Ecuador         Terremoto Manabi+6   
227 2016-04-23    Event  National     Ecuador         Terremoto Manabi+7   
228 2016-04-24    Event  National     Ecuador         Terremoto Manabi+8   
229 2016-04-25    Event  National     Ecuador         Terremoto Manabi+9   

     transferred  
219        False  
220        

All 12 detected outliers were investigated and confirmed to be legitimate 
real world events. The April 2016 spike corresponds to the Manabi earthquake 
which caused panic buying of groceries. The December spikes correspond to 
pre-Christmas shopping. Removing these would reduce model accuracy so we 
retain them and instead add holiday and earthquake indicators as features 
during feature engineering.
